# 12 - Full-record generator: 5 seeds, and the tail-dense dataset (run_02)

Same generator as notebook 11 v4 (ExGAN-MA-joint on the full record, spliced length-of-stay model, code repair,
procedure repeats from real stays). Two settings per seed:
- **matched**: extreme stays (above the 90th percentile of training PLOS stays) at their real share, about 10%.
  Used for the fidelity, privacy and consistency numbers, against the real-vs-real reference band.
- **tail_dense**: 30% extreme stays (`TAIL_SHARE = 0.30`, about 3 times the real share), all inside the observed range.
  This is the dataset that densifies the tail. Its overall length-of-stay metrics differ from real data **by design**,
  so it is judged on the extreme stays alone (Cell 8): do the generated extreme stays look like real extreme stays,
  and are they new patients rather than copies of the few real extreme patients?

Seeds 42-46. Seed 42 reuses the generator trained in notebook 11. Resumable: finished (seed, setting) runs are skipped.
Cell 9 writes the tail-dense dataset of seed 42 as Synthea files + notes.

Runtime: GPU (T4). About 10 minutes training per new seed, about 3 minutes evaluation per run; roughly 1 to 1.5 hours.

In [ ]:
# Cell 1 - settings
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
    sys.path.insert(0, "/content")
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
RAW, GENAI, ICD_MAP = f"{BASE}/raw/csv", f"{BASE}/genai", f"{BASE}/icd10cm/icd10cm_map.csv"
SEEDS = [42, 43, 44, 45, 46]
SETTINGS = {"matched": None, "tail_dense": 0.30}     # share of stays above the tail start; None = real share
N_SYN = 1000
K = {"ch": 50, "cs": 40, "pr": 40, "ma": 30, "mn": 40, "al": 15}
EPOCHS, BATCH, Z_DIM, HIDDEN = 150, 128, 64, 256
N_CRITIC, GP_W, LR, GUMBEL_TAU, LAMBDA_MODE = 5, 10.0, 2e-4, 0.2, 1.0
REPS, TAIL_Q, REPAIR, RETRAIN = 10, 0.90, True, False
if os.environ.get("SMOKE"): EPOCHS, N_SYN, REPS, SEEDS = 3, 300, 2, [42, 43]
CACHE, ROOT = f"{BASE}/synthetic_full/cache", f"{BASE}/synthetic_full"
RES = f"{ROOT}/results_12"; os.makedirs(RES, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
SEED, OUT = SEEDS[0], f"{ROOT}/seed{SEEDS[0]}"        # used by the shared cells below

In [ ]:
%%writefile /content/eval_suite.py
"""
eval_suite.py - evaluation suite for synthetic rare-class (tail) EHR data.

Compares a synthetic sample S against real data, following the literature review:
  - fidelity is measured against HELD-OUT real rare-class records (R), not the generator's training set (T)
  - every metric can be computed on a real-vs-real pair to get a reference value ("what good looks like")
  - privacy uses T (members) vs R (non-members)

Metric families (sources in docs/eval_methods.md):
  marginal   : KS (numeric), Wasserstein on the target (days), total variation (categorical/binary)
  association: mixed association matrix difference (Pearson / Cramer's V / correlation ratio)
  global     : discriminator AUC (classifier two-sample test), kernel MMD (RBF, median bandwidth)
  diversity  : precision / recall / density / coverage (Naeem et al. 2020), authenticity (Alaa et al. 2022)
  tail       : quantile error, exceedance error, expected shortfall error, GPD shape/scale difference
  subgroup   : mass error (overall and in the extreme tail), presence coverage, within-subgroup coverage
  privacy    : DCR ratio, NNDR, exact copies, distance-based membership inference AUC
  validity   : out-of-range numerics, unseen categories, target not above threshold
  utility    : augmentation gain in average precision with a paired patient-level bootstrap CI

All functions take pandas DataFrames with the same columns. No external services are called.
"""
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance, genpareto
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# direction used by the stress tests: +1 = larger is worse, -1 = smaller is worse, 0 = distance from an ideal value
# (ideal values in IDEAL); metrics not listed are informational
DIRECTION = {
    "ks_mean": 1, "target_wasserstein": 1, "tv_cat_mean": 1, "tv_cat_max": 1, "bin_prev_err_mean": 1, "bin_prev_err_max": 1,
    "assoc_diff_mean": 1, "disc_auc": 0, "mmd": 1,
    "precision": -1, "density": 0, "coverage": -1, "authenticity": -1,
    "tail_q_err_mean": 1, "exceed_err_mean": 1, "es90_rel_err": 1, "gpd_xi_diff": 1,
    "sub_mass_err_mean": 1, "sub_mass_err_max": 1, "sub_mass_err_extreme": 1, "sub_presence": -1,
    "sub_cov_mean": -1, "sub_cov_min": -1, "sub_min_share_ratio": -1,
    "dcr_ratio": -1, "dcr_p05_ratio": -1, "nndr_median": -1, "exact_copies": 1, "mia_auc": 1,
    "invalid_rate": 1,
}
IDEAL = {"disc_auc": 0.5, "density": 1.0}


class Schema:
    """Column roles. groups: {name: [binary flag columns]} for clinical subgroups (a row may be in several)."""
    def __init__(self, num, cat, bin_, target, threshold, groups=None, blocks=None):
        self.num, self.cat, self.bin, self.target = list(num), list(cat), list(bin_), target
        self.threshold, self.groups, self.blocks = threshold, (groups or {}), (blocks or {})
        self.cols = self.num + self.cat + self.bin


class Encoder:
    """Fit on the generator's training set T. Numeric -> z-score; categorical -> one-hot / sqrt(2) (a mismatch
    contributes distance 1, like one binary flag); binary -> 0/1. Optional blocks (lists of numeric columns,
    e.g. 128-d embeddings) are scaled so each block contributes like ONE feature, so embeddings do not drown
    out the table fields."""
    def __init__(self, schema, T):
        self.s = schema
        self.mu = T[schema.num].mean(); self.sd = T[schema.num].std().replace(0, 1)
        self.levels = {c: sorted(T[c].astype(str).unique()) for c in schema.cat}
        self.block_w = {}
        for b, cols in schema.blocks.items():
            self.block_w[b] = 1.0 / np.sqrt(max(len(cols), 1))

    def __call__(self, D):
        s = self.s
        z = ((D[s.num] - self.mu) / self.sd).astype(float)
        for b, cols in s.blocks.items():
            z[cols] = z[cols] * self.block_w[b]
        parts = [z.values]
        for c in s.cat:
            v = D[c].astype(str).values
            parts.append(np.stack([(v == l) for l in self.levels[c]], 1).astype(float) / np.sqrt(2))
        if s.bin:
            parts.append(D[s.bin].astype(float).values)
        return np.nan_to_num(np.concatenate(parts, 1))


# ---------------------------------------------------------------- marginals and associations
def _tv(a, b):
    p, q = a.astype(str).value_counts(normalize=True), b.astype(str).value_counts(normalize=True)
    idx = p.index.union(q.index)
    return 0.5 * (p.reindex(idx, fill_value=0) - q.reindex(idx, fill_value=0)).abs().sum()


def marginal_metrics(s, R, S):
    num_feats = [c for c in s.num if c not in sum(s.blocks.values(), [])]
    ks = [ks_2samp(R[c], S[c]).statistic for c in num_feats] or [np.nan]
    tv = [_tv(R[c], S[c]) for c in s.cat] or [np.nan]
    bp = (R[s.bin].astype(float).mean() - S[s.bin].astype(float).mean()).abs() if s.bin else pd.Series([np.nan])
    return {"ks_mean": float(np.mean(ks)), "target_wasserstein": wasserstein_distance(R[s.target], S[s.target]),
            "tv_cat_mean": float(np.mean(tv)), "tv_cat_max": float(np.max(tv)),
            "bin_prev_err_mean": float(bp.mean()), "bin_prev_err_max": float(bp.max())}


def _cramers_v(x, y):
    t = pd.crosstab(x, y).values
    if t.shape[0] < 2 or t.shape[1] < 2: return np.nan
    n = t.sum(); e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / n
    chi2 = ((t - e) ** 2 / np.where(e == 0, 1, e)).sum()
    return np.sqrt(chi2 / n / (min(t.shape) - 1))


def _corr_ratio(cat, num):
    num = num.astype(float)
    if num.std() == 0: return np.nan
    g = num.groupby(cat.astype(str).values)
    return np.sqrt((g.count() * (g.mean() - num.mean()) ** 2).sum() / ((num - num.mean()) ** 2).sum())


def association_matrix(s, D):
    """Mixed association: Pearson (numeric/binary pairs), Cramer's V (categorical with categorical/binary),
    correlation ratio (categorical with numeric). Embedding blocks are excluded (use MMD/discriminator there)."""
    emb = set(sum(s.blocks.values(), []))
    nums = [c for c in s.num if c not in emb] + s.bin
    cols = nums + s.cat
    M = pd.DataFrame(np.nan, index=cols, columns=cols)
    P = D[nums].astype(float).corr()
    M.loc[nums, nums] = P.values
    for c in s.cat:
        for o in cols:
            if o == c: continue
            if o in s.cat or o in s.bin:
                M.loc[c, o] = M.loc[o, c] = _cramers_v(D[c].astype(str), D[o].astype(str))
            else:
                M.loc[c, o] = M.loc[o, c] = _corr_ratio(D[c], D[o])
    return M


def association_metrics(s, R, S):
    A, B = association_matrix(s, R), association_matrix(s, S)
    d = (A - B).abs().values
    iu = np.triu_indices_from(d, 1)
    v = d[iu]
    # a pair constant in one sample and not the other counts as a full mismatch of its real strength
    v = np.where(np.isnan(v), np.nan_to_num(np.abs(A.values[iu])), v)
    return {"assoc_diff_mean": float(np.nanmean(v))}


# ---------------------------------------------------------------- global distribution tests
def discriminator_auc(XR, XS, seed=0, folds=5, gR=None, gS=None):
    """Classifier two-sample test: out-of-fold AUC of a real-vs-synthetic classifier on equal-size samples.
    0.5 = indistinguishable. Folds are grouped by patient (gR, gS = patient ids; None = every row its own
    group), so the classifier cannot score well by memorising a patient whose other stays are in training."""
    from sklearn.model_selection import StratifiedGroupKFold
    rng = np.random.default_rng(seed); n = min(len(XR), len(XS))
    iR, iS = rng.choice(len(XR), n, replace=False), rng.choice(len(XS), n, replace=False)
    XR, XS = XR[iR], XS[iS]
    gR = np.asarray(gR)[iR].astype(str) if gR is not None else np.array([f"r{i}" for i in iR])
    gS = np.asarray(gS)[iS].astype(str) if gS is not None else np.array([f"s{i}" for i in iS])
    X, y, g = np.vstack([XR, XS]), np.r_[np.zeros(n), np.ones(n)], np.r_[["R_" + x for x in gR], ["S_" + x for x in gS]]
    oof = np.zeros(len(y))
    for tr, te in StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(X, y, g):
        m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=seed).fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return roc_auc_score(y, oof)


def mmd_rbf(XR, XS, bandwidth, max_n=1500, seed=0):
    """Biased squared MMD with an RBF kernel; the bandwidth is fixed in advance from real data (median heuristic)."""
    rng = np.random.default_rng(seed)
    if len(XR) > max_n: XR = XR[rng.choice(len(XR), max_n, replace=False)]
    if len(XS) > max_n: XS = XS[rng.choice(len(XS), max_n, replace=False)]
    def k(A, B):
        d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
        return np.exp(-np.maximum(d2, 0) / (2 * bandwidth ** 2))
    return float(k(XR, XR).mean() + k(XS, XS).mean() - 2 * k(XR, XS).mean())


def median_bandwidth(X, max_n=1000, seed=0):
    rng = np.random.default_rng(seed)
    if len(X) > max_n: X = X[rng.choice(len(X), max_n, replace=False)]
    d = np.sqrt(np.maximum((X ** 2).sum(1)[:, None] + (X ** 2).sum(1)[None, :] - 2 * X @ X.T, 0))
    return float(np.median(d[np.triu_indices_from(d, 1)])) or 1.0


# ---------------------------------------------------------------- diversity (Naeem et al. 2020) and authenticity (Alaa et al. 2022)
def _kth_radius(X, k):
    d, _ = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)
    return d[:, k]  # column 0 is the point itself


def prdc(XR, XS, k=5):
    """Precision, recall, density, coverage with k-nearest-neighbour balls. Use equal sample sizes across models."""
    if len(XR) <= k or len(XS) <= k:
        return dict(precision=np.nan, recall=np.nan, density=np.nan, coverage=np.nan)
    rR, rS = _kth_radius(XR, k), _kth_radius(XS, k)
    D = np.sqrt(np.maximum((XS ** 2).sum(1)[:, None] + (XR ** 2).sum(1)[None, :] - 2 * XS @ XR.T, 0))  # syn x real
    return dict(precision=float((D < rR[None, :]).any(1).mean()),
                recall=float((D < rS[:, None]).any(0).mean()),
                density=float((D < rR[None, :]).sum(1).mean() / k),
                coverage=float((D.min(0) < rR).mean()))


def authenticity(XT, XS):
    """Share of synthetic rows farther from their nearest training row than that row is from its own nearest
    other training row (higher = less copy-like). A memorisation diagnostic, not a privacy guarantee."""
    nnT = NearestNeighbors(n_neighbors=2).fit(XT)
    dT = nnT.kneighbors(XT)[0][:, 1]
    d, i = NearestNeighbors(n_neighbors=1).fit(XT).kneighbors(XS)
    return float((d[:, 0] > dT[i[:, 0]]).mean())


# ---------------------------------------------------------------- privacy
def privacy_metrics(XT, XR, XS, seed=0):
    """DCR ratio = median distance (synthetic -> T) / median distance (held-out real R -> T); <1 means closer than
    unseen real patients. NNDR = d1/d2 to T. Membership inference: score each candidate by -distance to the
    synthetic set; members = T rows, non-members = R rows (equal sizes); AUC 0.5 = attacker at chance."""
    nnT = NearestNeighbors(n_neighbors=2).fit(XT)
    dS, _ = nnT.kneighbors(XS); dR, _ = nnT.kneighbors(XR)
    nndr = dS[:, 0] / np.where(dS[:, 1] == 0, np.nan, dS[:, 1])
    rng = np.random.default_rng(seed); n = min(len(XT), len(XR))
    mem, non = XT[rng.choice(len(XT), n, replace=False)], XR[rng.choice(len(XR), n, replace=False)]
    nnS = NearestNeighbors(n_neighbors=1).fit(XS)
    score = -np.r_[nnS.kneighbors(mem)[0][:, 0], nnS.kneighbors(non)[0][:, 0]]
    return {"dcr_ratio": float(np.median(dS[:, 0]) / max(np.median(dR[:, 0]), 1e-9)),
            "dcr_p05_ratio": float(np.quantile(dS[:, 0], .05) / max(np.quantile(dR[:, 0], .05), 1e-9)),
            "nndr_median": float(np.nanmedian(nndr)) if np.isfinite(nndr).any() else 0.0,
            "exact_copies": float((dS[:, 0] < 1e-6).mean()),
            "mia_auc": float(roc_auc_score(np.r_[np.ones(n), np.zeros(n)], score))}


# ---------------------------------------------------------------- tail (extreme value) fidelity
def tail_metrics(s, R, S, qs=(0.5, 0.75, 0.9, 0.95, 0.99)):
    y, z = R[s.target].astype(float).values, S[s.target].astype(float).values
    qerr = [abs(np.quantile(z, q) - np.quantile(y, q)) for q in qs]
    thr = [np.quantile(y, q) for q in (0.5, 0.75, 0.9, 0.95)]
    exc = [abs((z > v).mean() - (y > v).mean()) for v in thr]
    q90 = np.quantile(y, 0.9)
    es_r, es_s = y[y > q90].mean(), (z[z > q90].mean() if (z > q90).any() else np.nan)
    out = {"tail_q_err_mean": float(np.mean(qerr)), "exceed_err_mean": float(np.mean(exc)),
           "es90_rel_err": float(abs(es_s - es_r) / es_r) if np.isfinite(es_s) else 1.0}
    for q, e in zip(qs, qerr): out[f"q{int(q*100)}_err_days"] = float(e)
    try:
        xr = genpareto.fit(y[y > s.threshold] - s.threshold, floc=0)[0]
        xs = genpareto.fit(z[z > s.threshold] - s.threshold, floc=0)[0]
        out["gpd_xi_real"], out["gpd_xi_syn"], out["gpd_xi_diff"] = float(xr), float(xs), float(abs(xs - xr))
    except Exception:
        out["gpd_xi_diff"] = np.nan
    return out


# ---------------------------------------------------------------- subgroups (operational metrics recommended by both reviews)
def subgroup_metrics(s, R, S, XR, XS, k=5, k_min=5, extreme_q=0.9, min_extreme=20, S_full=None, real_shares=None):
    if not s.groups: return {}
    def mass(D):
        f = pd.DataFrame({g: D[c].astype(float).max(1) for g, c in s.groups.items()}, index=D.index)
        f["none"] = (f.sum(1) == 0).astype(float)
        return f
    fR, fS = mass(R), mass(S)
    err = (fR.mean() - fS.mean()).abs()
    cut = R[s.target].quantile(extreme_q)
    eR, eS = R[s.target] >= cut, S[s.target] >= cut
    ext = (fR[eR.values].mean() - fS[eS.values].mean()).abs().mean() if eR.sum() >= min_extreme and eS.sum() >= min_extreme else np.nan
    pres = np.mean([(fS[g] > 0).sum() >= k_min for g in s.groups])
    covs = []
    for g in s.groups:
        mR, mS = fR[g].values > 0, fS[g].values > 0
        kk = int(min(k, mR.sum() - 1, mS.sum() - 1))
        if kk >= 2:   # coverage needs a few neighbours on both sides; tiny groups are reported as NaN
            covs.append(prdc(XR[mR], XS[mS], k=kk)["coverage"])
    out = {"sub_mass_err_mean": float(err.mean()), "sub_mass_err_max": float(err.max()),
           "sub_mass_err_extreme": float(ext), "sub_presence": float(pres),
           "sub_cov_mean": float(np.mean(covs)) if covs else np.nan, "sub_cov_min": float(np.min(covs)) if covs else np.nan}
    for g in fR.columns: out[f"share_{g}"] = float(fS[g].mean())
    if S_full is not None and real_shares is not None:
        # smallest ratio (synthetic share / real share) over groups, on the FULL synthetic sample:
        # 0 = a subgroup is missing entirely, 1 = represented at its real rate
        fF = mass(S_full)
        out["sub_min_share_ratio"] = float(min(fF[g].mean() / max(real_shares[g], 1e-9) for g in s.groups))
    return out


# ---------------------------------------------------------------- validity
def validity_metrics(s, T, S, rules=None):
    """Rows breaking any rule: numeric (non-target) outside the training range, unseen category,
    target not above the rare-class threshold, plus optional custom rules (callables returning a bool mask)."""
    bad = np.zeros(len(S), bool)
    for c in [c for c in s.num if c != s.target and c not in sum(s.blocks.values(), [])]:
        bad |= ((S[c] < T[c].min()) | (S[c] > T[c].max())).values
    for c in s.cat:
        bad |= ~S[c].astype(str).isin(set(T[c].astype(str))).values
    bad |= (S[s.target] <= s.threshold).values
    for r in (rules or []):
        bad |= np.asarray(r(S), bool)
    return {"invalid_rate": float(bad.mean())}


# ---------------------------------------------------------------- one call for everything
def evaluate(s, T, R, S, enc=None, bandwidth=None, k=5, seed=0, rules=None, R_ids=None, S_ids=None, valid_ref=None):
    """T = generator training rows (real rare class), R = held-out real rare class, S = synthetic.
    Fidelity is S vs R; privacy is S vs T with R as non-member control. S is subsampled to len(R) for the
    sample-size-sensitive metrics (PRDC, MMD, discriminator) so every model is compared at the same size."""
    enc = enc or Encoder(s, T)
    rng = np.random.default_rng(seed)
    iq = rng.choice(len(S), min(len(S), len(R)), replace=False)
    S_eq = S.iloc[iq].reset_index(drop=True)
    S_ids_eq = np.asarray(S_ids)[iq] if S_ids is not None else None
    XT, XR, XS, XSe = enc(T), enc(R), enc(S), enc(S_eq)
    bandwidth = bandwidth or median_bandwidth(XR)
    out = {"n_syn": len(S)}
    out.update(marginal_metrics(s, R, S))
    out.update(association_metrics(s, R, S))
    out["disc_auc"] = discriminator_auc(XR, XSe, seed=seed, gR=R_ids, gS=S_ids_eq)
    out["mmd"] = mmd_rbf(XR, XSe, bandwidth, seed=seed)
    out.update(prdc(XR, XSe, k=k))
    out["authenticity"] = authenticity(XT, XS)
    out.update(tail_metrics(s, R, S))
    pooled = pd.concat([T, R], ignore_index=True)
    shares = {g: float(pooled[c].astype(float).max(1).mean()) for g, c in s.groups.items()}
    out.update(subgroup_metrics(s, R, S_eq, XR, XSe, k=k, S_full=S, real_shares=shares))
    out.update(privacy_metrics(XT, XR, XS, seed=seed))
    out.update(validity_metrics(s, valid_ref if valid_ref is not None else T, S, rules))
    return out


# ---------------------------------------------------------------- reference band and detection
def badness(name, value):
    """Turn a metric into 'larger = worse' so detection can use one rule."""
    d = DIRECTION.get(name)
    if d is None or value is None or not np.isfinite(value): return np.nan
    if d == 0: return abs(value - IDEAL[name])
    return value * d


def reference_band(s, T, real_pool, n_ref, reps=20, q=0.95, seed=0, patient_ids=None, **kw):
    """Evaluate `reps` real-vs-real pairs as if one side were synthetic. real_pool must be held out from T.
    The pool is split into two halves BY PATIENT (patient_ids), so one person's stays never sit on both sides.
    Returns the per-metric distribution; the q-quantile of 'badness' is the detection threshold."""
    rng = np.random.default_rng(seed); rows = []
    ids = np.asarray(patient_ids) if patient_ids is not None else np.arange(len(real_pool))
    uniq = np.unique(ids)
    for r in range(reps):
        half = set(rng.permutation(uniq)[:len(uniq) // 2])
        m = np.array([i in half for i in ids])
        R = real_pool[m].reset_index(drop=True)
        S = real_pool[~m].reset_index(drop=True)
        rows.append(evaluate(s, T, R, S, seed=seed + r, R_ids=ids[m], S_ids=ids[~m], **kw))
    ref = pd.DataFrame(rows)
    thr = {c: np.nanquantile([badness(c, v) for v in ref[c]], q) for c in ref.columns if c in DIRECTION}
    return ref, thr


def detected(result, thr, margin=0.0):
    return {c: bool(np.isfinite(badness(c, result.get(c, np.nan))) and badness(c, result[c]) > t + margin)
            for c, t in thr.items() if np.isfinite(t)}


# ---------------------------------------------------------------- utility: augmentation gain
def augmentation_gain(make_X, train, syn, val, test, label="is_plos", group="subject_id", p0=0.5,
                      n_boot=1000, seed=0, aug_weight=None):
    """Train the same classifier on real-only vs real+synthetic; evaluate both on the same real test patients.
    Average precision (AP) and recall at a precision floor p0 (threshold chosen on validation, then frozen).
    Paired bootstrap over test PATIENTS (all their stays together) gives a 95% CI for the AP gain."""
    Xtr, ytr = make_X(train), train[label].values
    Xsy = make_X(syn); Xva, yva = make_X(val), val[label].values; Xte, yte = make_X(test), test[label].values
    cols = Xtr.columns
    Xsy, Xva, Xte = (x.reindex(columns=cols, fill_value=0) for x in (Xsy, Xva, Xte))
    def fit(X, y):
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed).fit(X, y)
    m0 = fit(Xtr, ytr)
    if aug_weight == "balanced":   # baseline: real data only, class weighting instead of synthetic rows
        m1 = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed,
                                            class_weight="balanced").fit(Xtr, ytr)
    else:
        m1 = fit(pd.concat([Xtr, Xsy], ignore_index=True), np.r_[ytr, np.ones(len(Xsy), int)])
    out = {}
    preds = {}
    for name, m in [("real", m0), ("aug", m1)]:
        pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
        P, Rc, T = precision_recall_curve(yva, pv)
        ok = np.where(P[:-1] >= p0)[0]
        t = T[ok[np.argmax(Rc[:-1][ok])]] if len(ok) else np.inf
        out[f"ap_{name}"] = average_precision_score(yte, pt)
        out[f"recall_at_p{int(p0*100)}_{name}"] = float(((pt >= t) & (yte == 1)).sum() / max(yte.sum(), 1))
        out[f"precision_at_thr_{name}"] = float(yte[pt >= t].mean()) if (pt >= t).any() else np.nan
        preds[name] = pt
    out["ap_gain"] = out["ap_aug"] - out["ap_real"]
    rng = np.random.default_rng(seed)
    pats = test[group].values; up = np.unique(pats); pos = {p: np.where(pats == p)[0] for p in up}
    gains = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[p] for p in rng.choice(up, len(up), replace=True)])
        if yte[idx].sum() == 0: continue
        gains.append(average_precision_score(yte[idx], preds["aug"][idx]) - average_precision_score(yte[idx], preds["real"][idx]))
    out["ap_gain_ci_low"], out["ap_gain_ci_high"] = np.percentile(gains, [2.5, 97.5])
    return out


def summarize(raw, by="generator", metrics=None):
    """Mean, SD and a t-based 95% CI across independent generator seeds."""
    from scipy.stats import t as tdist
    metrics = metrics or [c for c in raw.columns if c not in (by, "seed") and pd.api.types.is_numeric_dtype(raw[c])]
    g = raw.groupby(by)[metrics]
    mean, sd, n = g.mean(), g.std(), g.count()
    half = sd * tdist.ppf(0.975, (n - 1).clip(lower=1)) / np.sqrt(n)
    return pd.concat({"mean": mean, "sd": sd, "ci_low": mean - half, "ci_high": mean + half}, axis=1)

In [ ]:
# Cell 3 - read the raw events of every cohort stay once (cached)
import json, math, time, uuid, pickle, tarfile, io, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
meta = json.load(open(f"{GENAI}/meta.json")); THR = meta["plos_threshold"]
CAT, BIN, NUM0 = meta["cat_cols"], meta["bin_cols"], meta["num_cols"]
MODES = {k: [c for c in v if c in BIN] for k, v in meta.get("modes", {}).items()}
MODES = {k: v for k, v in MODES.items() if v}
tab = pd.concat([pd.read_parquet(f"{GENAI}/{s}.parquet").assign(split=s) for s in ("train", "val", "test")], ignore_index=True)
tab["is_plos"] = (tab.target > THR).astype(int)
ids, subj = set(tab.hadm_id), set(tab.subject_id)
CH = 300_000
def header(f): return list(pd.read_csv(f"{RAW}/{f}", nrows=0).columns)
def read(f, cols, keep):
    cols = [c for c in cols if c in header(f)]
    return pd.concat([keep(c) for c in pd.read_csv(f"{RAW}/{f}", usecols=cols, dtype=str, chunksize=CH)], ignore_index=True)
def ts(s): return pd.to_datetime(s, errors="coerce", utc=True, format="ISO8601").dt.tz_localize(None)
VITALS = {"Heart rate": "hr", "Systolic Blood Pressure": "sbp", "Diastolic Blood Pressure": "dbp",
          "Respiratory rate": "rr", "Body Weight": "weight"}

if not os.path.exists(f"{CACHE}/events.pkl"):
    t0 = time.time()
    enc = read("encounters.csv", ["Id", "START", "STOP", "DESCRIPTION", "REASONCODE", "REASONDESCRIPTION"], lambda c: c[c.Id.isin(ids)])
    enc = enc.rename(columns={"Id": "hadm_id"}); enc["admit"] = ts(enc.START); enc["disch"] = ts(enc.STOP)
    adm = tab[["hadm_id", "subject_id"]].merge(enc[["hadm_id", "admit"]], on="hadm_id")
    def active_before(d):     # rows of the patient that started before admission and were not stopped by then
        d = d.rename(columns={"PATIENT": "subject_id"}).merge(adm, on="subject_id")
        d["START"] = ts(d.START); d["STOP"] = ts(d.STOP) if "STOP" in d else pd.NaT
        d = d[(d.START < d.admit) & (d.STOP.isna() | (d.STOP >= d.admit)) & (d.get("ENCOUNTER", "") != d.hadm_id)]
        return d.drop_duplicates(["hadm_id", "CODE"])[["hadm_id", "CODE", "DESCRIPTION"]]
    def scan(f, active):      # one chunked pass: rows of the stay itself + (optionally) rows active at admission
        if not os.path.exists(f"{RAW}/{f}"): return pd.DataFrame(columns=["hadm_id", "CODE", "DESCRIPTION"]), None
        cols = [c for c in ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"] if c in header(f)]
        stay, act = [], []
        for c in pd.read_csv(f"{RAW}/{f}", usecols=cols, dtype=str, chunksize=CH):
            stay.append(c[c.ENCOUNTER.isin(ids)].rename(columns={"ENCOUNTER": "hadm_id"}))
            if active:
                c = c[c.PATIENT.isin(subj)]
                if len(c): act.append(active_before(c))
        st = pd.concat(stay, ignore_index=True)
        return st, (pd.concat(act, ignore_index=True).drop_duplicates(["hadm_id", "CODE"]) if act else pd.DataFrame(columns=["hadm_id", "CODE", "DESCRIPTION"]))
    cs, ch = scan("conditions.csv", True); cs = cs.drop_duplicates(["hadm_id", "CODE"])[["hadm_id", "CODE", "DESCRIPTION"]]
    pr, _ = scan("procedures.csv", False)
    mn, ma = scan("medications.csv", True); mn = mn.drop_duplicates(["hadm_id", "CODE"])[["hadm_id", "CODE", "DESCRIPTION"]]
    _, al = scan("allergies.csv", True)
    ob = read("observations.csv", ["ENCOUNTER", "DATE", "CODE", "DESCRIPTION", "VALUE", "UNITS"],
              lambda c: c[c.ENCOUNTER.isin(ids) & c.DESCRIPTION.isin(VITALS)]).rename(columns={"ENCOUNTER": "hadm_id"})
    ob["VALUE"] = pd.to_numeric(ob.VALUE, errors="coerce"); ob["ch"] = ob.DESCRIPTION.map(VITALS)
    vit = ob.groupby(["hadm_id", "ch"]).VALUE.agg(["mean", "std"]).unstack()
    vit_lookup = ob.drop_duplicates("ch").set_index("ch")[[c for c in ("CODE", "DESCRIPTION", "UNITS") if c in ob]].to_dict("index")
    pickle.dump(dict(enc=enc, cs=cs, ch=ch, pr=pr, mn=mn, ma=ma, al=al, vit=vit, vit_lookup=vit_lookup), open(f"{CACHE}/events.pkl", "wb"))
    print(f"raw files read in {time.time() - t0:.0f}s")
EV = pickle.load(open(f"{CACHE}/events.pkl", "rb"))
for k in ("cs", "ch", "pr", "mn", "ma", "al"):
    if EV[k] is None: EV[k] = pd.DataFrame(columns=["hadm_id", "CODE", "DESCRIPTION"])   # e.g. no allergies.csv in this export
    EV[k]["CODE"] = EV[k]["CODE"].astype(str)
print(f"Stays {len(tab):,} | PLOS {tab.is_plos.mean():.1%} | events: " +
      ", ".join(f"{k} {len(EV[k]):,}" for k in ("ch", "cs", "pr", "ma", "mn", "al")))

In [ ]:
# Cell 4 - one wide row per stay: tabular + vitals + counts + multi-hot codes (top codes chosen on TRAIN PLOS stays)
icd = pd.read_csv(ICD_MAP, dtype=str).set_index("snomed")["icd"].to_dict()
plos_tr = set(tab.loc[(tab.split == "train") & (tab.is_plos == 1), "hadm_id"]); train_ids = set(tab.loc[tab.split == "train", "hadm_id"])
GROUP_NAME = {"ch": "condition history", "cs": "stay conditions", "pr": "procedures", "ma": "medications active at admission",
              "mn": "medications started in stay", "al": "allergies"}
DESC, MH, coverage = {}, {}, []
GROUP_NAME = {g: n for g, n in GROUP_NAME.items() if len(EV[g])}       # skip groups with no data
for g in GROUP_NAME:
    e = EV[g]
    for c, d in e.groupby("CODE").DESCRIPTION.agg(lambda s: s.value_counts().index[0]).items(): DESC[(g, c)] = d
    rank = pd.concat([e[e.hadm_id.isin(plos_tr)].drop_duplicates(["hadm_id", "CODE"]).CODE.value_counts(),
                      e[e.hadm_id.isin(train_ids)].drop_duplicates(["hadm_id", "CODE"]).CODE.value_counts() * 1e-6]).groupby(level=0).sum()
    top = list(rank.sort_values(ascending=False).index[:K[g]])
    MH[g] = top
    ep = e[e.hadm_id.isin(plos_tr)]
    coverage.append({"group": GROUP_NAME[g], "codes kept": len(top), "codes seen": e.CODE.nunique(),
                     "share of PLOS occurrences covered": ep.CODE.isin(top).mean() if len(ep) else np.nan})
print(pd.DataFrame(coverage).round(3).to_string(index=False))

D = tab.copy().set_index("hadm_id")
VCH = ["hr", "sbp", "dbp", "rr", "weight"]; VCOLS = [f"vital_{c}" for c in VCH]
vit = EV["vit"]
for c in VCH:
    D[f"vital_{c}"] = vit[("mean", c)].reindex(D.index) if ("mean", c) in vit else np.nan
D["has_vitals"] = D[VCOLS].notna().any(axis=1).astype(int)
VMED = D.loc[D.split == "train", VCOLS].median()
D[VCOLS] = D[VCOLS].fillna(VMED)
VSD = {c: float(vit[("std", c)].median()) if ("std", c) in vit else 0.0 for c in VCH}   # typical within-stay spread
D["n_proc"] = EV["pr"].groupby("hadm_id").size().reindex(D.index).fillna(0)
D["n_hist_cond"] = EV["ch"].groupby("hadm_id").size().reindex(D.index).fillna(0)
COUNTS = ["n_proc", "n_hist_cond"]
MH_COLS = []
for g, top in MH.items():
    x = EV[g][EV[g].CODE.isin(top)].drop_duplicates(["hadm_id", "CODE"])
    W = pd.crosstab(x.hadm_id, x.CODE).reindex(index=D.index, columns=top, fill_value=0).clip(0, 1)
    W.columns = [f"{g}_{c}" for c in top]; MH_COLS += list(W.columns)
    D = D.join(W)
D[MH_COLS] = D[MH_COLS].fillna(0).astype(int)
# procedure repeat rate per day of stay, where present (train stays)
pc = EV["pr"][EV["pr"].hadm_id.isin(train_ids)].groupby(["hadm_id", "CODE"]).size().rename("n").reset_index()
pc = pc.merge(tab[["hadm_id", "target"]], on="hadm_id")
PCNT = {c: (g.target.values.astype(float), g.n.values) for c, g in pc.groupby("CODE")}   # per procedure: (stay length, times done) in real training stays

NUMS = NUM0 + VCOLS + COUNTS
BINS = BIN + ["has_vitals"] + MH_COLS
GEN_COLS = NUMS + ["target"] + CAT + BINS
D = D.reset_index()
for c in CAT: D[c] = D[c].astype(str)
train, val, test = (D[D.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
print(f"\nRecord width: {len(NUMS)} numeric + target + {len(CAT)} categorical + {len(BINS)} binary ({len(MH_COLS)} code flags)")

In [ ]:
# Cell 5 - codec and networks (from notebook 11), plus train / generate as functions
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import genpareto
DEV = "cuda" if torch.cuda.is_available() else "cpu"
def seed_all(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    return np.random.default_rng(s)
MODE_NAMES = list(MODES); NM = len(MODE_NAMES)
def mode_matrix(d):
    if NM == 0: return np.zeros((len(d), 0), np.float32)
    return np.stack([d[MODES[k]].max(axis=1).values for k in MODE_NAMES], 1).astype(np.float32)

class Codec:
    def __init__(self, df):
        self.log = set(COUNTS)
        v = self._num(df); self.mu, self.sd = v.mean(0), v.std(0) + 1e-6
        self.lo, self.hi = df[NUMS].min().values, df[NUMS].max().values
        logl = np.log(df["target"].clip(lower=1)); self.l_mu, self.l_sd = logl.mean(), logl.std()
        self.levels = {c: sorted(df[c].astype(str).unique()) for c in CAT}
        self.cat_sizes = [len(self.levels[c]) for c in CAT]; self.n_num, self.n_bin = len(NUMS), len(BINS)
        self.dim = self.n_num + sum(self.cat_sizes) + 2 * self.n_bin
    def _num(self, df):
        return np.stack([np.log1p(df[c].values.astype(float)) if c in COUNTS else df[c].values.astype(float) for c in NUMS], 1)
    def encode_x(self, df):
        parts = [(self._num(df) - self.mu) / self.sd]
        for c in CAT:
            v = df[c].astype(str).values; parts.append(np.stack([(v == l) for l in self.levels[c]], 1))
        b = df[BINS].values.astype(np.float32); parts.append(np.stack([1 - b, b], 2).reshape(len(df), -1))
        return np.concatenate(parts, 1).astype(np.float32)
    def encode_e(self, los): return ((np.log(np.clip(los, 1, None)) - self.l_mu) / self.l_sd).astype(np.float32)[:, None]
    def decode(self, x, los):
        n = x[:, :self.n_num] * self.sd + self.mu
        out = {}
        for j, c in enumerate(NUMS):
            v = np.expm1(n[:, j]) if c in COUNTS else n[:, j]
            v = np.clip(v, self.lo[j], self.hi[j])
            out[c] = np.round(v) if (c in COUNTS or c == "anchor_age") else v
        out["target"] = np.round(los) if INT_LOS else los; i = self.n_num
        for c, s in zip(CAT, self.cat_sizes):
            out[c] = np.array(self.levels[c])[x[:, i:i + s].argmax(1)]; i += s
        flags = x[:, i:].reshape(len(x), self.n_bin, 2).argmax(2)
        return pd.concat([pd.DataFrame(out), pd.DataFrame(flags, columns=BINS)], axis=1)[GEN_COLS]
INT_LOS = bool((train.target % 1 == 0).all())
codec = Codec(train)
cat_start = codec.n_num; bin_start = codec.n_num + sum(codec.cat_sizes)
mode_bin_idx = [[BINS.index(c) for c in MODES[k]] for k in MODE_NAMES]

class Generator(nn.Module):
    def __init__(self, n_cond):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(Z_DIM + n_cond, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, codec.dim))
    def forward(self, z, c):
        h = self.net(torch.cat([z, c], 1)); outs, i = [h[:, :codec.n_num]], codec.n_num
        for s in codec.cat_sizes: outs.append(F.gumbel_softmax(h[:, i:i + s], tau=GUMBEL_TAU)); i += s
        outs.append(F.gumbel_softmax(h[:, i:].reshape(-1, codec.n_bin, 2), tau=GUMBEL_TAU, dim=2).reshape(len(h), -1))
        return torch.cat(outs, 1)
class Critic(nn.Module):
    def __init__(self, n_cond):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(codec.dim + n_cond, HIDDEN), nn.LeakyReLU(0.2), nn.Linear(HIDDEN, HIDDEN),
                                 nn.LeakyReLU(0.2), nn.Linear(HIDDEN, 1))
    def forward(self, x, c): return self.net(torch.cat([x, c], 1))
def grad_penalty(D_, real, fake, c):
    a = torch.rand(len(real), 1, device=DEV); mix = (a * real + (1 - a) * fake).requires_grad_(True)
    g = torch.autograd.grad(D_(mix, c).sum(), mix, create_graph=True)[0]
    return ((g.norm(2, dim=1) - 1) ** 2).mean()
def mode_probs(x):
    p1 = x[:, bin_start + 1::2]
    return torch.stack([p1[:, idx].max(1).values for idx in mode_bin_idx], 1)

X_all, E_all, M_train = codec.encode_x(train), codec.encode_e(train.target.values.astype(float)), mode_matrix(train)
pat = pd.Series(["".join(map(str, r.astype(int))) for r in M_train]) if NM else pd.Series(["x"] * len(train))
freq = pat.map(pat.value_counts()).values.astype(float); row_w = np.log(freq + 1) / freq
row_w = torch.tensor(row_w / row_w.sum(), dtype=torch.float)


def train_or_load(seed, out):
    os.makedirs(out, exist_ok=True)
    rng = seed_all(seed)
    G, Dn = Generator(1 + NM).to(DEV), Critic(1 + NM).to(DEV)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9)); optD = torch.optim.Adam(Dn.parameters(), lr=LR, betas=(0.5, 0.9))
    Xt, Ct = torch.tensor(X_all, device=DEV), torch.tensor(np.concatenate([E_all, M_train], 1), device=DEV)
    Mt = torch.tensor(M_train, device=DEV); steps = math.ceil(len(Xt) / BATCH); t0 = time.time()
    LOADED = (not RETRAIN) and os.path.exists(f"{out}/generator.pt")
    if LOADED: G.load_state_dict(torch.load(f"{out}/generator.pt", map_location=DEV)); print("Loaded the trained generator from generator.pt")
    for ep in range(0 if LOADED else EPOCHS):
        for _ in range(steps):
            idx = torch.multinomial(row_w, BATCH, replacement=True).to(DEV); real, c = Xt[idx], Ct[idx]
            for _ in range(N_CRITIC):
                fake = G(torch.randn(BATCH, Z_DIM, device=DEV), c).detach()
                lossD = Dn(fake, c).mean() - Dn(real, c).mean() + GP_W * grad_penalty(Dn, real, fake, c)
                optD.zero_grad(); lossD.backward(); optD.step()
            fake = G(torch.randn(BATCH, Z_DIM, device=DEV), c); lossG = -Dn(fake, c).mean()
            if NM: lossG = lossG + LAMBDA_MODE * F.binary_cross_entropy(mode_probs(fake).clamp(1e-6, 1 - 1e-6), Mt[idx])
            optG.zero_grad(); lossG.backward(); optG.step()
        if (ep + 1) % 25 == 0 or ep + 1 == EPOCHS: print(f"  epoch {ep + 1}/{EPOCHS} | lossD {lossD.item():.3f} | {time.time() - t0:.0f}s", flush=True)
    torch.save(G.state_dict(), f"{out}/generator.pt")

    return G

def generate(G, rng, tail_share, out, verbose=False):
    global train
    # length of stay from the GPD tail; modes drawn given length of stay (the "joint" variant)
    real_tail = train[train.is_plos == 1].reset_index(drop=True)
    # Spliced length-of-stay model: real stays below the TAIL_Q quantile (with +/-0.5 day jitter), a GPD above it,
    # truncated at the longest real training stay so nothing is generated outside the observed range.
    y = real_tail.target.values.astype(float); U, YMAX = np.quantile(y, TAIL_Q), y.max()
    xi, _, sigma = genpareto.fit(y[y > U] - U, floc=0)
    share = (y > U).mean() if tail_share is None else tail_share
    n_t = rng.binomial(N_SYN, share)
    body = np.clip(rng.choice(y[y <= U], N_SYN - n_t) + rng.uniform(-0.5, 0.5, N_SYN - n_t), THR + 1e-3, U)
    Fmax = genpareto.cdf(YMAX - U, xi, scale=sigma)
    tail = U + genpareto.ppf(rng.random(n_t) * Fmax, xi, scale=sigma)
    los = rng.permutation(np.r_[body, tail])
    if INT_LOS: los = np.clip(np.round(los), math.floor(THR) + 1, YMAX)
    M_tail = mode_matrix(real_tail)
    edges = np.unique(np.quantile(real_tail.target, np.linspace(0, 1, 11)))
    rb = np.clip(np.searchsorted(edges, real_tail.target, side="right") - 1, 0, len(edges) - 2)
    qb = np.clip(np.searchsorted(edges, los, side="right") - 1, 0, len(edges) - 2)
    m_req = np.zeros((N_SYN, NM), np.float32)
    for b in np.unique(qb):
        pool = np.where(rb == b)[0]; pool = pool if len(pool) else np.arange(len(real_tail))
        m_req[qb == b] = M_tail[rng.choice(pool, (qb == b).sum())]
    with torch.no_grad():
        G.eval(); xs = G(torch.randn(N_SYN, Z_DIM, device=DEV), torch.tensor(np.concatenate([codec.encode_e(los), m_req], 1), device=DEV)).cpu().numpy()
    S = codec.decode(xs, los)
    S.loc[S.has_vitals == 0, VCOLS] = VMED.values          # same rule as the real data: no vitals -> training median
    REASON = next((c for c in CAT if "reason" in c.lower()), None)
    if REPAIR:
        key = REASON or "_mode_key"
        if not REASON:
            train["_mode_key"] = ["".join(map(str, r.astype(int))) for r in M_train]; S["_mode_key"] = ["".join(map(str, r.astype(int))) for r in mode_matrix(S)]
        before = {}
        for g in ("cs", "pr", "mn"):
            cols = [c for c in MH_COLS if c.startswith(g + "_")]
            if not cols: continue
            A = train.groupby(key)[cols].max().reindex(S[key].astype(str)).fillna(0).values
            before[g] = int(S[cols].values.sum()); S[cols] = (S[cols].values * A).astype(int)
            if verbose: print(f"  repair {GROUP_NAME[g]}: kept {S[cols].values.sum():,} of {before[g]:,} generated codes")
        S = S.drop(columns=["_mode_key"], errors="ignore"); train = train.drop(columns=["_mode_key"], errors="ignore")
    # counts must agree with the codes, as they do in real data (no procedure codes -> 0 procedures)
    # how many times each procedure is done: taken from real training stays with that procedure and the closest
    # length of stay (random one of the 5 nearest), so repeats follow the real pattern per procedure
    PCOUNTS = []
    for los_i, row in zip(S.target.values, S[[c for c in MH_COLS if c.startswith("pr_")]].values):
        d = {}
        for col, on in zip([c for c in MH_COLS if c.startswith("pr_")], row):
            if on:
                L, Nn = PCNT[col[3:]]; near = np.argsort(np.abs(L - los_i))[:5]; d[col[3:]] = int(Nn[rng.choice(near)])
        PCOUNTS.append(d)
    S["n_proc"] = [sum(d.values()) for d in PCOUNTS]
    S["n_hist_cond"] = np.maximum(S.n_hist_cond.values, S[[c for c in MH_COLS if c.startswith("ch_")]].sum(1).values)
    S.to_parquet(f"{out}/records.parquet", index=False)
    info = {"tail_start_days": U, "n_tail": int(n_t), "gpd_xi": xi, "max_los": float(los.max()), "mode_consistency": float((mode_matrix(S) == m_req).all(1).mean()) if NM else np.nan}
    return S, PCOUNTS, info

In [ ]:
# Cell 6 - fixed references: real-vs-real band (whole PLOS sample) and the real extreme stays
import importlib, eval_suite as ev; importlib.reload(ev)
from scipy.stats import spearmanr
BLOCKS = {g: [c for c in MH_COLS if c.startswith(g + "_")] for g in MH}
schema = ev.Schema(num=NUMS + MH_COLS + ["target"], cat=CAT, bin_=BIN + ["has_vitals"], target="target", threshold=THR,
                   groups=MODES, blocks=BLOCKS)
COLS = schema.cols
T = train[train.is_plos == 1][COLS].reset_index(drop=True)
R = val[val.is_plos == 1][COLS].reset_index(drop=True); P = test[test.is_plos == 1][COLS].reset_index(drop=True)
R_IDS = val.loc[val.is_plos == 1, "subject_id"].values
POOL_IDS = np.r_[R_IDS, test.loc[test.is_plos == 1, "subject_id"].values]
enc = ev.Encoder(schema, T); BW = ev.median_bandwidth(enc(R)); VALID_REF = train[COLS]
ref_path = f"{RES}/reference_band.csv"
if os.path.exists(ref_path): ref = pd.read_csv(ref_path)
else:
    ref, _ = ev.reference_band(schema, T, pd.concat([R, P], ignore_index=True), n_ref=len(R), reps=REPS,
                               patient_ids=POOL_IDS, enc=enc, bandwidth=BW, valid_ref=VALID_REF)
    ref.to_csv(ref_path, index=False)
# extreme stays: above the 90th percentile of training PLOS stays
U = float(np.quantile(T.target, TAIL_Q))
pool_all = pd.concat([R, P], ignore_index=True)
R_ext, T_ext = pool_all[pool_all.target > U].reset_index(drop=True), T[T.target > U].reset_index(drop=True)
R_ext_ids = POOL_IDS[(pool_all.target > U).values]
print(f"T {len(T)} | R {len(R)} | tail starts at {U:.1f} d | real extreme stays: training {len(T_ext)}, held-out {len(R_ext)}")

In [ ]:
# Cell 7 - run every (seed, setting): generate, evaluate, save one row each (resumable)
SEEN = {(ga, gb): set() for ga, gb in [("cs", "mn"), ("cs", "pr"), ("ch", "ma")]}
def pair_list(df, ga, gb):
    A, B = [c for c in MH_COLS if c.startswith(ga + "_")], [c for c in MH_COLS if c.startswith(gb + "_")]
    a, b = df[A].values.astype(bool), df[B].values.astype(bool)
    return [(i, j) for r in range(len(df)) for i in np.where(a[r])[0] for j in np.where(b[r])[0]]
for k in SEEN: SEEN[k] = set(pair_list(train, *k))
def consistency(S):
    out = {}
    for (ga, gb), seen in SEEN.items():
        p = pair_list(S, ga, gb); out[f"pairs_seen_{ga}_{gb}"] = np.mean([x in seen for x in p]) if p else np.nan
    for g in MH:
        cols = BLOCKS[g]
        out[f"prev_err_{g}"] = float((S[cols].mean() - R[cols].mean()).abs().mean()); out[f"codes_per_rec_{g}"] = S[cols].sum(1).mean()
    out["spearman_nproc_los"] = spearmanr(S.n_proc, S.target)[0]
    for q in (0.9, 0.95, 0.99): out[f"share_above_T_q{int(q*100)}"] = (S.target > np.quantile(T.target, q)).mean()
    out["max_los"] = S.target.max()
    return out
def tail_eval(S, rng, reps=5):
    """Generated extreme stays vs held-out real extreme stays; training extreme stays vs the same, as the real reference."""
    S_ext = S[S.target > U][COLS].reset_index(drop=True); n = len(R_ext); out = {"n_syn_extreme": len(S_ext)}
    XR, XT = enc(R_ext), enc(T_ext)
    k = int(min(5, n - 1, len(T_ext) - 1))
    vals = {"syn": [], "train_real": []}
    for r in range(reps):
        Ss = S_ext.sample(min(n, len(S_ext)), random_state=int(rng.integers(10**9)))
        Ts = T_ext.sample(min(n, len(T_ext)), random_state=int(rng.integers(10**9)))
        for name, D in (("syn", Ss), ("train_real", Ts)):
            X = enc(D); m = {"disc_auc": ev.discriminator_auc(XR, X, seed=r, gR=R_ext_ids), "mmd": ev.mmd_rbf(XR, X, BW)}
            m.update({f"{kk}": v for kk, v in ev.prdc(XR, X, k=k).items() if kk in ("precision", "coverage")})
            m["code_prev_err"] = float(np.mean([(D[BLOCKS[g]].mean() - R_ext[BLOCKS[g]].mean()).abs().mean() for g in MH]))
            m["mode_share_err"] = float(np.mean([abs(D[MODES[g]].max(1).mean() - R_ext[MODES[g]].max(1).mean()) for g in MODES])) if MODES else np.nan
            m["los_q_err"] = float(np.mean([abs(np.quantile(D.target, q) - np.quantile(R_ext.target, q)) for q in (0.25, 0.5, 0.75, 0.9)]))
            vals[name].append(m)
    for name in vals:
        for kk, v in pd.DataFrame(vals[name]).mean().items(): out[f"ext_{kk}_{name}"] = v
    # privacy in the tail: are generated extreme stays closer to training patients than unseen real extreme stays are?
    XTall = enc(T); nn = ev.NearestNeighbors(n_neighbors=1).fit(XTall)
    dS = nn.kneighbors(enc(S_ext))[0][:, 0]; dR = nn.kneighbors(XR)[0][:, 0]
    out["ext_dcr_ratio"] = float(np.median(dS) / np.median(dR))
    out["ext_near_copy_share"] = float((dS < np.quantile(dR, 0.05)).mean())    # closer than 95% of unseen real extreme stays
    out["ext_exact_copies"] = float((dS < 1e-8).mean())
    return out

LOG = f"{RES}/runs.csv"
done = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame(columns=["seed", "setting"])
for seed in SEEDS:
    out = f"{ROOT}/seed{seed}"
    todo = [s for s in SETTINGS if not ((done.seed == seed) & (done.setting == s)).any()]
    if not todo: print(f"seed {seed}: done"); continue
    G = train_or_load(seed, out)
    for j, setting in enumerate(SETTINGS):
        if setting not in todo: continue
        t0 = time.time(); rng = np.random.default_rng(seed * 10 + j)
        os.makedirs(f"{out}/{setting}", exist_ok=True)
        S, PCOUNTS, info = generate(G, rng, SETTINGS[setting], f"{out}/{setting}")
        row = {"seed": seed, "setting": setting, **info}
        row.update(ev.evaluate(schema, T, R, S[COLS], enc=enc, bandwidth=BW, R_ids=R_IDS, valid_ref=VALID_REF, seed=seed))
        row.update(consistency(S)); row.update(tail_eval(S, rng))
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(LOG, index=False)
        pickle.dump(PCOUNTS, open(f"{out}/{setting}/pcounts.pkl", "wb"))
        print(f"seed {seed} {setting:10s}: disc {row['disc_auc']:.3f} | ks {row['ks_mean']:.3f} | extreme stays {row['n_syn_extreme']} "
              f"| extreme disc {row['ext_disc_auc_syn']:.3f} (real {row['ext_disc_auc_train_real']:.3f}) | {time.time() - t0:.0f}s", flush=True)

In [ ]:
# Cell 8a - matched setting across seeds: mean, SD and 95% CI next to the real-vs-real band
from scipy.stats import t as tdist
runs = pd.read_csv(LOG)
def summ(d, cols):
    n = len(d); m, sd = d[cols].mean(), d[cols].std(ddof=1)
    h = tdist.ppf(0.975, max(n - 1, 1)) * sd / np.sqrt(n)
    return pd.DataFrame({"mean": m, "sd": sd, "ci_low": m - h, "ci_high": m + h})
keys = [k for k in ev.DIRECTION if k in runs and k in ref]
M = summ(runs[runs.setting == "matched"], keys)
M["real-vs-real 5%"], M["real-vs-real median"], M["real-vs-real 95%"] = ref[keys].quantile(0.05), ref[keys].median(), ref[keys].quantile(0.95)
M["mean inside band"] = (M["mean"] >= M["real-vs-real 5%"]) & (M["mean"] <= M["real-vs-real 95%"])
M.to_csv(f"{RES}/matched_summary.csv")
print(f"Matched setting, {runs[runs.setting == 'matched'].seed.nunique()} seeds")
M.round(3)

In [ ]:
# Cell 8b - consistency (matched vs tail_dense) and the extreme stays (the tail-densification result)
ccols = [c for c in runs if c.startswith(("pairs_seen", "prev_err", "codes_per_rec", "spearman", "share_above", "max_los"))]
real_c = consistency(pd.concat([R], ignore_index=True)); real_c = pd.Series(real_c)[ccols]
C = pd.concat({s: summ(runs[runs.setting == s], ccols)["mean"] for s in SETTINGS}, axis=1); C.insert(0, "real held-out", real_c)
print("Consistency (mean over seeds):"); print(C.round(3).to_string())
ext = ["disc_auc", "mmd", "precision", "coverage", "code_prev_err", "mode_share_err", "los_q_err"]
E = pd.DataFrame({
    "training extreme stays (real reference)": runs[[f"ext_{m}_train_real" for m in ext]].mean().values,
    **{f"{s}: generated extreme stays": runs.loc[runs.setting == s, [f"ext_{m}_syn" for m in ext]].mean().values for s in SETTINGS},
    **{f"{s}: SD over seeds": runs.loc[runs.setting == s, [f"ext_{m}_syn" for m in ext]].std(ddof=1).values for s in SETTINGS}},
    index=[f"{m} vs held-out extreme stays" for m in ext])
P_ = runs.groupby("setting")[["n_syn_extreme", "ext_dcr_ratio", "ext_near_copy_share", "ext_exact_copies"]].mean()
E.to_csv(f"{RES}/extreme_stays.csv"); P_.to_csv(f"{RES}/extreme_privacy.csv")
print(f"\nExtreme stays (above {U:.1f} d): generated vs {len(R_ext)} held-out real extreme stays")
print(E.round(3).to_string())
print("\nPrivacy in the tail (DCR ratio about 1 and near-copy share about 0.05 = like unseen real patients):")
print(P_.round(3).to_string())

In [ ]:
# Cell 9 - write the tail-dense dataset of the first seed as Synthea files + notes
def write_dataset(S, PCOUNTS, OUT, rng):
    lo_t, hi_t = EV["enc"].admit.min(), EV["enc"].admit.max()
    hours = EV["enc"].admit.dt.hour.values
    def pick(col_name_part):
        for c in CAT:
            if col_name_part in c.lower(): return c
    reason_col, type_col = pick("reason"), pick("type")
    gcol, rcol, ecol = pick("gender"), pick("race"), pick("ethnic")
    real_pat = pd.read_csv(f"{RAW}/patients.csv", usecols=lambda c: c in ("GENDER", "RACE", "ETHNICITY"), dtype=str)
    enc_real = EV["enc"]
    reason_code = enc_real.dropna(subset=["REASONDESCRIPTION"]).groupby("REASONDESCRIPTION").REASONCODE.agg(lambda s: s.mode().iloc[0]).to_dict() \
                  if "REASONCODE" in enc_real else {}
    enc_type = enc_real.DESCRIPTION.mode().iloc[0]
    def u(): return str(uuid.UUID(bytes=rng.bytes(16), version=4))
    def days(x): return pd.Timedelta(days=float(x))
    def codes(row, g): return [c.split("_", 1)[1] for c in MH_COLS if c.startswith(g + "_") and row[c] == 1]
    T_PAT, T_ENC, T_CON, T_PRO, T_MED, T_ALG, T_OBS, NOTES = ([] for _ in range(8))
    for i, r in S.iterrows():
        pid, hid = u(), u()
        admit = (lo_t + (hi_t - lo_t) * rng.random()).normalize() + pd.Timedelta(hours=int(rng.choice(hours)), minutes=int(rng.integers(60)))
        los = float(r.target); disch = admit + days(los)
        sex = r[gcol] if gcol else real_pat.GENDER.sample(1, random_state=int(rng.integers(10**9))).iloc[0]
        race = r[rcol] if rcol else real_pat.RACE.sample(1, random_state=int(rng.integers(10**9))).iloc[0]
        eth = r[ecol] if ecol else real_pat.ETHNICITY.sample(1, random_state=int(rng.integers(10**9))).iloc[0]
        first = "Jane" if str(sex).upper().startswith("F") else "John"
        T_PAT.append({"Id": pid, "BIRTHDATE": (admit - days(365.25 * r.anchor_age + rng.integers(0, 365))).date(), "FIRST": first,
                      "LAST": "Doe", "GENDER": sex, "RACE": race, "ETHNICITY": eth})
        stay, hist = codes(r, "cs"), codes(r, "ch")
        if reason_col and r[reason_col] in reason_code: rc, rd = reason_code[r[reason_col]], r[reason_col]
        elif stay: rc, rd = stay[0], DESC[("cs", stay[0])]
        else: rc, rd = "", ""
        if rc and rc not in stay and ("cs", rc) in DESC: stay = [rc] + stay
        T_ENC.append({"Id": hid, "START": admit, "STOP": disch, "PATIENT": pid, "ENCOUNTERCLASS": "inpatient",
                      "DESCRIPTION": r[type_col] if type_col else enc_type, "REASONCODE": rc, "REASONDESCRIPTION": rd})
        for c in hist:
            T_CON.append({"START": (admit - days(rng.integers(30, 3650))).date(), "PATIENT": pid, "ENCOUNTER": "", "CODE": c,
                          "DESCRIPTION": DESC[("ch", c)], "ICD10CM": icd.get(c, "")})
        for c in stay:
            T_CON.append({"START": admit.date(), "PATIENT": pid, "ENCOUNTER": hid, "CODE": c,
                          "DESCRIPTION": DESC.get(("cs", c), ""), "ICD10CM": icd.get(c, "")})
        # procedures: the generated total is split over the present procedures by their real per-day rate (at least 1 each)
        pcs = codes(r, "pr"); proc_list = []
        if pcs:
            for c in pcs:
                k = PCOUNTS[i][c]
                for t in np.sort(rng.random(k)) * los:
                    T_PRO.append({"START": admit + days(t), "STOP": admit + days(t) + pd.Timedelta(minutes=30), "PATIENT": pid,
                                  "ENCOUNTER": hid, "CODE": c, "DESCRIPTION": DESC[("pr", c)]})
                    proc_list.append((t, DESC[("pr", c)]))
        act, new = codes(r, "ma"), codes(r, "mn")
        for c in act:
            T_MED.append({"START": admit - days(rng.integers(30, 1500)), "STOP": "", "PATIENT": pid, "ENCOUNTER": "", "CODE": c, "DESCRIPTION": DESC[("ma", c)]})
        for c in new:
            T_MED.append({"START": admit + days(rng.random() * min(los, 2)), "STOP": disch, "PATIENT": pid, "ENCOUNTER": hid, "CODE": c, "DESCRIPTION": DESC[("mn", c)]})
        algs = codes(r, "al")
        for c in algs:
            T_ALG.append({"START": (admit - days(rng.integers(365, 7300))).date(), "PATIENT": pid, "ENCOUNTER": "", "CODE": c, "DESCRIPTION": DESC[("al", c)]})
        if r.has_vitals == 1:
            for d in range(int(min(max(los, 1), 60))):
                for ch_ in VCH:
                    lk = EV["vit_lookup"].get(ch_, {})
                    T_OBS.append({"DATE": admit + pd.Timedelta(days=d, hours=float(rng.random() * 12)), "PATIENT": pid, "ENCOUNTER": hid,
                                  "CATEGORY": "vital-signs", "CODE": lk.get("CODE", ""), "DESCRIPTION": lk.get("DESCRIPTION", ch_),
                                  "VALUE": round(float(r[f"vital_{ch_}"] + rng.normal(0, VSD[ch_] or 0)), 1), "UNITS": lk.get("UNITS", ""), "TYPE": "numeric"})
        # note, following Synthea's note.ftl section by section
        lc = lambda xs: [str(x).lower() for x in xs]
        hpi = f"{first} Doe is a {int(r.anchor_age)} year-old {eth} {race} {'female' if first == 'Jane' else 'male'}."
        if hist: hpi += " Patient has a history of " + ", ".join(lc(DESC[("ch", c)] for c in hist)) + "."
        meds_active = lc([DESC[("ma", c)] for c in act] + [DESC[("mn", c)] for c in new])
        plan = ""
        if proc_list: plan += "The following procedures were conducted:\n" + "".join(f"- {d.lower()}\n" for _, d in sorted(proc_list))
        if new: plan += "The patient was prescribed the following medications:\n" + "".join(f"- {m}\n" for m in lc(DESC[("mn", c)] for c in new))
        note = (f"{admit:%Y-%m-%d}\n\n# Chief Complaint\nNo complaints.\n\n# History of Present Illness\n{hpi}\n\n"
                f"# Social History\nPatient has never smoked.\nPatient currently has {'Medicare' if r.anchor_age >= 65 else 'private insurance'}.\n\n"
                f"# Allergies\n{', '.join(lc(DESC[('al', c)] for c in algs)) or 'No Known Allergies.'}\n\n"
                f"# Medications\n{'; '.join(meds_active) or 'No Active Medications.'}\n\n# Assessment and Plan\n"
                + (f"Patient is presenting with {', '.join(lc(DESC.get(('cs', c), '') for c in stay))}. " if stay else "")
                + f"\n\n## Plan\n\n{plan}")
        NOTES.append((pid, hid, note))

    def write(rows, fname):
        real_cols = header(fname) if os.path.exists(f"{RAW}/{fname}") else []
        df = pd.DataFrame(rows)
        cols = real_cols + [c for c in df.columns if c not in real_cols]
        df.reindex(columns=cols).to_csv(f"{OUT}/{fname}", index=False); return len(df)
    os.makedirs(OUT, exist_ok=True)
    for rows, f in [(T_PAT, "patients.csv"), (T_ENC, "encounters.csv"), (T_CON, "conditions.csv"), (T_PRO, "procedures.csv"),
                    (T_MED, "medications.csv"), (T_ALG, "allergies.csv"), (T_OBS, "observations.csv")]:
        print(f"{f:18s} {write(rows, f):>8,} rows")
    with tarfile.open(f"{OUT}/notes.tar.gz", "w:gz") as tf:
        for pid, _, note in NOTES:
            b = note.encode(); ti = tarfile.TarInfo(f"notes/Synthetic_Doe_{pid}.txt"); ti.size = len(b); tf.addfile(ti, io.BytesIO(b))
    print(f"notes.tar.gz       {len(NOTES):>8,} notes  ->  {OUT}")
s0 = SEEDS[0]; d0 = f"{ROOT}/seed{s0}/tail_dense"
S0 = pd.read_parquet(f"{d0}/records.parquet"); PC0 = pickle.load(open(f"{d0}/pcounts.pkl", "rb"))
write_dataset(S0, PC0, d0, np.random.default_rng(s0))
print(f"Extreme stays in this dataset: {(S0.target > U).sum()} of {len(S0)} (real share {(T.target > U).mean():.0%})")